<div style="border-bottom: 4px solid #003366; margin-bottom: 20px; padding-bottom: 10px; display: flex; justify-content: space-between; align-items: center;">
    <div style="flex-grow: 1;">
        <h1 style="color: #003366; font-family: 'Helvetica', sans-serif; margin-bottom: 5px;">MAT2605: Cálculo Científico I</h1>
        <h2 style="color: #555; margin-top: 0; margin-bottom: 10px;">Laboratorio 04: M&eacute;todos iterativos</h2>
        <p style="margin: 2px 0;"><b>Profesores:</b> Thomas F&uuml;hrer y Manuel A. Sánchez | <b>Fecha:</b> 10 de Abril, 2026</p>
        <p style="margin: 2px 0;"><b>Institución:</b> Facultad de Matemáticas, Pontificia Universidad Católica de Chile</p>
    </div>
    <div style="flex: 0 0 auto; margin-left: 20px;">
        <img src="../source/FacMatematicas-15.png" 
             alt="Logo UC" 
             style="height: 60px; width: auto;">
    </div>

</div>

<div style="border: 1px solid #2980b9; border-left: 8px solid #2980b9; padding: 15px; border-radius: 5px; margin-bottom: 20px;">
    <h3 style="color: #2980b9; margin-top: 0;">🎯 Objetivos de la Sesión</h3>
    <ul>
        <li>M&eacute;todos de Jacobi y Gauss-Seidel</li>
        <li>M&eacute;todo de Richardson y gradient descent</li>
        <li>Normas vectoriales inducidas por matrices</li>
    </ul>
</div>

<div style="background-color: #e8f5e9; border-left: 5px solid #4caf50; padding: 15px; border-radius: 5px; margin-bottom: 20px;">
    <h3 style="color: #2e7d32; margin-top: 0;">👤 Identificación del Estudiante</h3>
    <p style="margin-bottom: 10px; color: #555;">Por favor, completa tus datos antes de comenzar:</p>
    <ul style="list-style-type: none; padding-left: 0; color: #333;">
        <li style="margin-bottom: 5px;"><b>📌 Nombre Completo:</b> _______________________________________</li>
        <li style="margin-bottom: 5px;"><b>📌 Rol / Nº Alumno:</b> _________________________</li>
    </ul>
    <p style="font-size: 0.85em; color: #2e7d32; margin-top: 15px; border-top: 1px solid #a5d6a7; padding-top: 5px;">
        <i>💡 Haz doble clic en esta celda para editarla con tus datos.</i>
    </p>
</div>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# INTENTO DE CONFIGURACIÓN DE ESTILO (Compatible con versiones viejas y nuevas)
try:
    plt.style.use('seaborn-v0_8-whitegrid') # Nombre nuevo (Matplotlib 3.6+)
except OSError:
    plt.style.use('seaborn-whitegrid')      # Nombre antiguo

plt.rcParams['figure.figsize'] = (10, 6)

print(f"Versión de Numpy: {np.__version__}")
print(f"Estilo usado: {plt.style.context}")

Versión de Numpy: 2.4.4
Estilo usado: <function context at 0x7ffb3d5ea350>


<div class="alert-block alert-warning "; style=" border-left: 5px solid #ffc107; padding: 15px; border-radius: 5px;">
    <h2 > </h2>
    <h3 style="color: #d35400; margin-top: 0;">📚 Teoría:</h3>
    <p>Sea $A\in\mathbb{R}^{N\times N}$ una matriz definida positiva, $A=A^\top$ y todos sus valores propios son positivos.
        Se puede verificar que $(x,y)_A := (Ax,y)_2 = \sum_{j=1}^N (Ax)_jy_j$ ($x,y\in\mathbb{R}^N$) es producto escalar y $\|x\|_A = \sqrt{(x,x)_A}$ define norma vectorial.
    </p>
    <p>M&eacute;todo iterativo b&aacute;sico</p>
    $$ x^{(k+1)} = T x^{(k)} + b, \quad x^{(0)} \text{ vector inicial dado}$$
    donde $T$ es la matriz de iteraci&oacute;n. Para asegurar convergencia es suficiente que $\rho(T)<1$ donde $\rho$ es el radio espectral.
    <p>M&eacute;todo de Jacobi: Escribimos $A = D-L-U$ donde $D$ es la parte diagonal, $-L$, $-U$ son las partes triangulares (inferior y superior). Se define $T_J = D^{-1}(L+U)$</p>
    <p>M&eacute;todo de Gauss--Seidel: $T_{GS} = (D-L)^{-1}U$</p>
    <p>M&eacute;todo de Richardson: $T_{R} = (I-\alpha A)$. El m&eacute;todo converge si se elige $\alpha \in (0,2\lambda_{\mathrm{max}}(A))$ y $A$ es definida positiva</p>
</div>

# M&eacute;todo de Jacobi / Gauss-Seidel / SOR / Richardson

Abajo se presentan los codigos de Jacobi, Gauss-Seidel y SOR

In [2]:
def metodo_de_Jacobi(A, b, x0, max_iter=100, tol=1e-8):
    """
    Resuelve el sistema Ax = b usando el método de Jacobi.
    Input: A, b, x0 (vector inicial)
    Output: x (solución aproximada), residuals (historial de errores)
    """
    x = x0.copy()
    n = len(b)
    residuals = []
    
    for k in range(max_iter):
        # Calcular el residuo actual
        residual = b - np.dot(A, x)
        norm_residual = np.linalg.norm(residual)
        residuals.append(norm_residual)
        
        # Criterio de parada
        if norm_residual < tol:
            print(f"J: Convergencia alcanzada en la iteración {k+1} | Error: {norm_residual:.6e}")
            return x, residuals
        x_old = x.copy()
        # Calcular los nuevos valores de x
        for i in range(n):
            suma = np.dot(A[i, :i], x_old[:i]) + np.dot(A[i, i+1:], x_old[i+1:])
            # Jacobi
            x[i] = (b[i] - suma) / A[i, i]
            
    print(f"J: Número máximo de iteraciones alcanzado. Error final: {norm_residual:.6e}")
    return x, residuals

In [3]:
def metodo_de_GaussSeidel(A, b, x0, max_iter=100, tol=1e-8):
    """
    Resuelve el sistema Ax = b usando el método de Gauss Seidel.
    Input: A, b
    Output: x
    """
    x = x0.copy()
    
    residuals = []
    n= x.size
    
    for k in range(max_iter):
        residual = b - np.dot(A, x)
        norm_residual = np.linalg.norm(residual)
        residuals.append(norm_residual)

        # print(f"   {k+1}      | {norm_residual:.6f}")

        if norm_residual < tol:
            print(f"   {k+1}      | {norm_residual:.6f}")
            print("\nConvergencia alcanzada.")
            return x, residuals
        xnew = np.zeros(n)
        for i in range(n):
            if i == 0:
                xnew[i] = (1.0/A[i,i])*(b[i] -  A[i,i+1:].dot(x[i+1:]) )
            elif i == n-1:
                xnew[i] = (1.0/A[i,i])*(b[i] -  A[i,:i+1].dot(xnew[:i+1]) )
            else:
                xnew[i] = (1.0/A[i,i])*(b[i] - A[i,:i].dot(xnew[:i]) -  A[i,i+1:].dot(x[i+1:])  )
        
        x[:] = xnew[:]
    print(f"GS: Número máximo de iteraciones alcanzado. Error final: {norm_residual:.6e}")
    return x, residuals

In [4]:
def metodo_de_SOR(A, b, x0, w=1.0, max_iter=100, tol=1e-8):
    """
    Resuelve el sistema Ax = b usando el método de SOR.
    Input: A, b
    Output: x
    """
    x = x0.copy(); n= x.size
    residuals = []
    for k in range(max_iter):
        residual = b - np.dot(A, x)
        norm_residual = np.linalg.norm(residual)
        residuals.append(norm_residual)
        if norm_residual < tol:
            print(f"   {k+1}      | {norm_residual:.6f}")
            print("\nConvergencia alcanzada.")
            return x, residuals
        xnew = np.zeros(n)
        for i in range(n):
            if i == 0:
                xnew[i] = (1.0-w)*x[i] + (w*1.0/A[i,i])*(b[i] -  A[i,i+1:].dot(x[i+1:]) )
            elif i == n-1:
                xnew[i] = (1.0-w)*x[i] + (w*1.0/A[i,i])*(b[i] -  A[i,:i+1].dot(xnew[:i+1]) )
            else:
                xnew[i] = (1.0-w)*x[i] + (w*1.0/A[i,i])*(b[i] - A[i,:i].dot(xnew[:i]) -  A[i,i+1:].dot(x[i+1:])  )
        x[:] = xnew[:]
    print(f"SOR: Número máximo de iteraciones alcanzado. Error final: {norm_residual:.6e}")
    return x, residuals

In [5]:
def metodo_de_Richardson(A, b, x0, w=1.0, max_iter=100, tol=1e-8):
    """
    Resuelve el sistema Ax = b usando el método de Richardson
    Input: A, b
    Output: x
    """
    x = x0.copy(); n= x.size
    residuals = []
    for k in range(max_iter):
        residual = b - np.dot(A, x)
        norm_residual = np.linalg.norm(residual)
        residuals.append(norm_residual)
        if norm_residual < tol:
            print(f"   {k+1}      | {norm_residual:.6f}")
            print("\nConvergencia alcanzada.")
            return x, residuals
        x = x + w*residual
        
    print(f"R: Número máximo de iteraciones alcanzado. Error final: {norm_residual:.6e}")
    return x, residuals

## Ejemplo 1: 

\begin{equation}
A = 
\begin{bmatrix}
1 & 1/2 & 1/2 \\
1/2 & 1 & 1/2 \\
1/2 & 1/2 & 1
\end{bmatrix}
\end{equation}

Nota: La matriz es definida positiva.

In [6]:
A = np.array([[1.0, 0.5,0.5],[0.5,1.0,0.5],[0.5,0.5,1.0]])
b = np.array([1.0,-.1,0.0])

print('\nMetodo Jacobi')
x = metodo_de_Jacobi(A,b,np.ones(3))[0]

print('\nMetodo Gauss-Seidel')
x = metodo_de_GaussSeidel(A,b,np.ones(3))[0]

print('\nMetodo SOR con parametro w = 1')
x = metodo_de_SOR(A,b,np.ones(3),1.0)[0]

print('\nMetodo de Richardson con parametro w = 0.8')
x = metodo_de_Richardson(A,b,np.ones(3),0.8)[0]


Metodo Jacobi
J: Número máximo de iteraciones alcanzado. Error final: 2.944486e+00

Metodo Gauss-Seidel
   20      | 0.000000

Convergencia alcanzada.

Metodo SOR con parametro w = 1
   20      | 0.000000

Convergencia alcanzada.

Metodo de Richardson con parametro w = 0.8
   40      | 0.000000

Convergencia alcanzada.


Nota: Las funciones <code>metodo_de_Jacobi</code> tienen 2 outputs: x y residuals. 
Si estamos interesados solo en la primera variable podemos escribir <code>nombre_funcion(sus,argument,os)[0]</code>
(ver los ejemplos arriba)

In [7]:
print('\nMetodo de Richardson con parametro w = 0.8')
x = metodo_de_Richardson(A,b,np.ones(3),0.8)[0]

xExacto = np.linalg.solve(A,b)

print('Norma euclideana del error: ',np.linalg.norm(x-xExacto))


Metodo de Richardson con parametro w = 0.8
   40      | 0.000000

Convergencia alcanzada.
Norma euclideana del error:  5.044888968908186e-09


# Ejemplo 2 (Collatz):

\begin{equation}
A = 
\left(
\begin{array}{ccc}
1 & 2 & -2 \\
1 & 1 & 1 \\
2 & 2 & 1
\end{array}
\right)
\end{equation}

Convergen los m&eacute;todos?

In [8]:
A = np.array([[1,2,-2],[1,1,1],[2,2,1]],dtype = np.float64)
D = np.diag(np.diag(A))
L = -np.tril(A, k=-1)
U = -np.triu(A, k=1)

### M&eacute;todo de Jacobi

In [9]:
# Matriz e iteracion de Jacobi
T_J = np.linalg.inv(D)@ (L+U)

print("T_J:\n",T_J)
# % spectral radius condition
rho_J = max(abs(np.linalg.eigvals(T_J)))
print(" rho_J = ", rho_J, "==> converge!")

T_J:
 [[ 0. -2.  2.]
 [-1.  0. -1.]
 [-2. -2.  0.]]
 rho_J =  1.0812771377650137e-05 ==> converge!


### M&eacute;todo de Gauss-Seidel

In [10]:
# Matriz e iteracion de Gauss-Seidel
T_G = np.linalg.inv(D-L)@ (U)
print("T_G:\n",T_G)
# % spectral radius condition
rho_G = max(abs(np.linalg.eigvals(T_G)))
print(" rho_G = ", rho_G, "==> no converge!")

T_G:
 [[ 0. -2.  2.]
 [ 0.  2. -3.]
 [ 0.  0.  2.]]
 rho_G =  2.0 ==> no converge!


### Método de Richardson

In [11]:
# Matriz e iteracion de Richardson
w = 1
T_R = np.identity(3)-w*A
print("T_R:\n",T_R)
# % spectral radius condition
rho_R = max(abs(np.linalg.eigvals(T_R)))
print(" rho_R = ", rho_R, "==> converge!")

T_R:
 [[ 0. -2.  2.]
 [-1.  0. -1.]
 [-2. -2.  0.]]
 rho_R =  1.0812771377650137e-05 ==> converge!


---
<div class="alert alert-block alert-success"> 

## Evaluaci&oacute;n:

</div>

<div style="background-color: #e8f5e9; border-left: 5px solid #4caf50; padding: 15px; border-radius: 5px;">
    <h3 style="color: #2e7d32; margin-top: 0;">💻 Ejercicio 1:</h3>
    Se anuncia durante el Laboratorio.
</div>

<div style="background-color: #e8f5e9; border-left: 5px solid #4caf50; padding: 15px; border-radius: 5px;">
    <h3 style="color: #6a1b9a; margin: 0; font-size: 1.1em;">
        ✏️ Solución / Desarrollo
    </h3>
    <p style="margin: 5px 0 0 0; font-size: 0.9em; color: #666;">
        <i>Utilice las celdas de código y texto debajo de esta línea para responder. Recuerde comentar su código.</i>
    </p>
</div>

In [32]:
A = np.array([[1,2,3],[4,5,4],[3,2,1]])
b = np.array([1,1,1])
x = np.linalg.solve(A,b) # solucion exacta

# vector inicial
x0 = np.zeros(3)

# Tu codigo:



<div style="background-color: #e8f5e9; border-left: 5px solid #4caf50; padding: 15px; border-radius: 5px;">
    <h3 style="color: #2e7d32; margin-top: 0;">💻 Ejercicio 2:</h3>
Se anuncia durante el Laboratorio.
</div>

<div style="background-color: #e8f5e9; border-left: 5px solid #4caf50; padding: 15px; border-radius: 5px;">
    <h3 style="color: #6a1b9a; margin: 0; font-size: 1.1em;">
        ✏️ Solución / Desarrollo
    </h3>
    <p style="margin: 5px 0 0 0; font-size: 0.9em; color: #666;">
        <i>Utilice las celdas de código y texto debajo de esta línea para responder. Recuerde comentar su código.</i>
    </p>
</div>

<div style="background-color: #e8f5e9; border-left: 5px solid #4caf50; padding: 15px; border-radius: 5px;">
    <h3 style="color: #2e7d32; margin-top: 0;">💻 Ejercicio 3: </h3>
    Se anuncia durante el Laboratorio.
</div>

<div style="background-color: #e8f5e9; border-left: 5px solid #4caf50; padding: 15px; border-radius: 5px;">
    <h3 style="color: #6a1b9a; margin: 0; font-size: 1.1em;">
        ✏️ Solución / Desarrollo
    </h3>
    <p style="margin: 5px 0 0 0; font-size: 0.9em; color: #666;">
        <i>Utilice las celdas de código y texto debajo de esta línea para responder. Recuerde comentar su código.</i>
    </p>
</div>